# Level 4 - Clean Architecture RAG with `ragentkit`

This notebook tests the brand new `ragentkit` library (imported as `raglib`).
It uses our Azure dataset and runs an advanced pipeline using Clean Architecture principles.

In [ ]:
import os
import json
import asyncio
import nest_asyncio
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter

# raglib components
from raglib import EmbedderFactory, VectorStoreFactory, RAG
from raglib.domain import Chunk, Query
from raglib.shared.result import Ok

# Apply nest_asyncio to allow async execution inside Jupyter Notebooks
nest_asyncio.apply()

load_dotenv()

## 1. Load and Chunk Dataset
We reuse our 1000-character chunking strategy.

In [ ]:
with open('dataset.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

raglib_chunks = []
chunk_id_counter = 1

for item in data:
    if 'content' not in item:
        continue
    
    splits = text_splitter.split_text(item['content'])
    for split in splits:
        # Convert LangChain texts into raglib.domain.Chunk models
        chunk = Chunk(
            id=f"chunk_{chunk_id_counter}",
            content=split,
            metadata={"title": item.get('title', 'Unknown'), "url": item.get('url', 'Unknown')}
        )
        raglib_chunks.append(chunk)
        chunk_id_counter += 1

print(f"Created {len(raglib_chunks)} raglib Chunks.")

## 2. Initialize raglib Infrastructure (Async)
raglib uses an advanced asynchronous `Result[Ok, Err]` pattern for maximum stability.

In [ ]:
async def setup_raglib_pipeline():
    print("Initializing Embedder...")
    embedder = EmbedderFactory.create("sentence-transformer")
    
    print("Generating Embeddings for Chunks (This might take a minute)...")
    texts = [c.content for c in raglib_chunks]
    embeddings_result = await embedder.embed_documents(texts)
    
    if not isinstance(embeddings_result, Ok):
        print("Failed to embed documents:", embeddings_result)
        return None
        
    embeddings = embeddings_result.value
    
    print("Loading into In-Memory Vector Store...")
    store = VectorStoreFactory.create("in-memory")
    await store.aadd_chunks(raglib_chunks, embeddings)
    
    print("Quickstarting RAG Agent with Gemini...")
    # Build the full Graph Engine Pipeline!
    agent = RAG.quickstart(store=store, llm="gemini", embedder=embedder)
    print("Agent Ready!")
    
    return agent

# Run the async setup
agent = asyncio.run(setup_raglib_pipeline())

## 3. Query the Pipeline

In [ ]:
async def test_query(agent):
    if not agent:
        print("Agent failed to initialize.")
        return
        
    print("\n=== Testing raglib Pipeline ===")
    query = Query(text="What is Azure Chaos Studio?")
    print(f"Query: {query.text}")
    
    # Execute the query through the compiled graph
    response = await agent.ainvoke(query)
    
    print("\n=== Answer ===")
    print(response.content)

asyncio.run(test_query(agent))